# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbottabad123/flyrank-ml-track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
# Section 1: Two paper findings + methodology questions

import pandas as pd

paper_findings = pd.DataFrame({
    "Finding": [
        "30–59 days of search visibility had a 3.59:1 growth-to-decline ratio",
        "The growth prediction model reported about 90% accuracy on known brands and 75% on unseen brands"
    ],
    "Methodology Question": [
        "How exactly is the growth/decline label created, and which pages are included or excluded from the ratio?",
        "Does the validation split keep unseen brands completely separate from training, and is the label based only on information available after the feature window?"
    ]
})

display(paper_findings)


,Finding,Methodology Question
0,30–59 days of search visibility had a 3.59:1 g...,How exactly is the growth/decline label create...
1,The growth prediction model reported about 90%...,Does the validation split keep unseen brands c...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# Section 2: My model under an honest split
# This cell is completely independent.

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score


# -----------------------------------
# Recreate Week-5 dataset
# -----------------------------------

X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.60, 0.40],
    class_sep=1.2,
    random_state=42
)

X = pd.DataFrame(
    X_raw,
    columns=[
        "feature_1",
        "feature_2",
        "feature_3",
        "feature_4",
        "feature_5",
        "feature_6",
        "feature_7",
        "feature_8"
    ]
)

y = pd.Series(y_raw, name="target")


# -----------------------------------
# Model
# -----------------------------------

def make_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])


# -----------------------------------
# BEFORE: Random split
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

before_model = make_model()

before_model.fit(
    X_train,
    y_train
)

before_pred = before_model.predict(X_test)

before_prob = before_model.predict_proba(
    X_test
)[:, 1]

before_accuracy = accuracy_score(
    y_test,
    before_pred
)

before_auc = roc_auc_score(
    y_test,
    before_prob
)


# -----------------------------------
# AFTER: Grouped split
# -----------------------------------

# Synthetic groups of 10 rows
groups = np.arange(len(X)) // 10

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

after_model = make_model()

after_model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

after_pred = after_model.predict(
    X.iloc[test_idx]
)

after_prob = after_model.predict_proba(
    X.iloc[test_idx]
)[:, 1]

after_accuracy = accuracy_score(
    y.iloc[test_idx],
    after_pred
)

after_auc = roc_auc_score(
    y.iloc[test_idx],
    after_prob
)


# -----------------------------------
# BEFORE vs AFTER comparison
# -----------------------------------

comparison = pd.DataFrame({
    "Split": [
        "Before - Random Split",
        "After - Grouped Stress Split"
    ],
    "Accuracy": [
        before_accuracy,
        after_accuracy
    ],
    "ROC-AUC": [
        before_auc,
        after_auc
    ]
})

display(comparison.round(3))


# -----------------------------------
# Group separation check
# -----------------------------------

overlap = set(
    groups[train_idx]
).intersection(
    set(groups[test_idx])
)

print("Train/Test groups overlap:", len(overlap) > 0)
print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))

,Split,Accuracy,ROC-AUC
0,Before - Random Split,0.830,0.919
1,After - Grouped Stress Split,0.835,0.924


Train/Test groups overlap: False
Training rows: 800
Testing rows: 200


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# Section 3: Leakage Audit

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


# -----------------------------------
# Recreate Week-5 dataset
# -----------------------------------

X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.60, 0.40],
    class_sep=1.2,
    random_state=42
)

X = pd.DataFrame(
    X_raw,
    columns=[
        "feature_1",
        "feature_2",
        "feature_3",
        "feature_4",
        "feature_5",
        "feature_6",
        "feature_7",
        "feature_8"
    ]
)

y = pd.Series(y_raw, name="target")


# -----------------------------------
# 1. Feature-set audit
# -----------------------------------

print("LEAKAGE AUDIT")
print("=" * 50)

print("\nFeature columns:")
print(list(X.columns))

print("\nTarget column:")
print(y.name)

# Check target leakage
target_in_features = y.name in X.columns

print(
    "\nTarget included in features:",
    target_in_features
)

# Check ID-like columns
id_columns = [
    column for column in X.columns
    if column.lower() in ["id", "client_id"]
    or column.lower().endswith("_id")
]

print(
    "ID-like columns:",
    id_columns
)

print(
    "Total features:",
    len(X.columns)
)


# -----------------------------------
# 2. Check suspicious feature names
# -----------------------------------

leakage_words = [
    "target",
    "label",
    "outcome",
    "future",
    "result",
    "prediction",
    "score",
    "trend"
]

suspicious_features = [
    column for column in X.columns
    if any(word in column.lower() for word in leakage_words)
]

print(
    "\nPotentially suspicious feature names:",
    suspicious_features
)


# -----------------------------------
# 3. Deliberate leakage test
# -----------------------------------

# Add target directly as a fake feature.
# A leakage test should produce extremely high performance.

X_leaky = X.copy()

X_leaky["target_copy"] = y.values


X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)


leaky_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


leaky_model.fit(
    X_train,
    y_train
)


leaky_probability = leaky_model.predict_proba(
    X_test
)[:, 1]


leaky_auc = roc_auc_score(
    y_test,
    leaky_probability
)


print(
    "\nDeliberate target-copy leakage ROC-AUC:",
    round(leaky_auc, 3)
)


# -----------------------------------
# 4. Final leakage decision
# -----------------------------------

print("\nFINAL AUDIT")
print("=" * 50)

if target_in_features:
    print("WARNING: Target leakage detected.")
else:
    print("PASS: Target is not included in the final features.")

if len(suspicious_features) == 0:
    print("PASS: No suspicious feature names detected.")
else:
    print(
        "REVIEW: Suspicious features found:",
        suspicious_features
    )

print(
    "The deliberate target_copy feature is NOT used in the final model."
)

print(
    "\nConclusion: No obvious label leakage was found "
    "in the available synthetic feature set."
)

LEAKAGE AUDIT

Feature columns:
['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8']

Target column:
target

Target included in features: False
ID-like columns: []
Total features: 8

Potentially suspicious feature names: []

Deliberate target-copy leakage ROC-AUC: 1.0

FINAL AUDIT
PASS: Target is not included in the final features.
PASS: No suspicious feature names detected.
The deliberate target_copy feature is NOT used in the final model.

Conclusion: No obvious label leakage was found in the available synthetic feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [7]:
# Section 4: Claim Rewrite

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score


# -----------------------------------
# Recreate Week-5 dataset
# -----------------------------------

X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.60, 0.40],
    class_sep=1.2,
    random_state=42
)

X = pd.DataFrame(
    X_raw,
    columns=[f"feature_{i}" for i in range(1, 9)]
)

y = pd.Series(y_raw, name="target")


# -----------------------------------
# Create model
# -----------------------------------

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# -----------------------------------
# Grouped stress split
# -----------------------------------

groups = np.arange(len(X)) // 10

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)


# -----------------------------------
# Train model
# -----------------------------------

model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)


# -----------------------------------
# Predictions
# -----------------------------------

predictions = model.predict(
    X.iloc[test_idx]
)

probabilities = model.predict_proba(
    X.iloc[test_idx]
)[:, 1]


# -----------------------------------
# Metrics
# -----------------------------------

accuracy = accuracy_score(
    y.iloc[test_idx],
    predictions
)

auc = roc_auc_score(
    y.iloc[test_idx],
    probabilities
)


# -----------------------------------
# Failure examples
# -----------------------------------

audit = X.iloc[test_idx].copy()

audit["actual"] = y.iloc[test_idx].values
audit["predicted"] = predictions
audit["predicted_probability"] = probabilities

errors = audit[
    audit["actual"] != audit["predicted"]
]

print("Grouped-test rows:", len(audit))
print("Grouped-test errors:", len(errors))

print("\nExample model failures:")
display(errors.head(10).round(3))


# -----------------------------------
# Claim rewrite
# -----------------------------------

print("\n" + "=" * 70)
print("CLAIM REWRITE")
print("=" * 70)

print("""
TOO-STRONG CLAIM:

"The model accurately predicts the target and generalizes well."


EVIDENCE-SAFE CLAIM:

"Observed and measured: the model achieved {:.3f} accuracy
and {:.3f} ROC-AUC on the grouped stress-test split.

The result was directionally stable compared with the original
random split. These measurements support the model as
decision-support on this synthetic evaluation.

However, the results do not establish generalization to unseen
clients or future production data because the Week-5 dataset
does not contain real client IDs or timestamps."
""".format(accuracy, auc))

Grouped-test rows: 200
Grouped-test errors: 33

Example model failures:


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,actual,predicted,predicted_probability
2,-3.971,-0.428,0.283,1.621,-1.244,1.067,-0.958,0.450,0,1,0.712
45,0.456,-1.495,1.299,0.265,-0.412,-2.521,-0.358,-0.739,0,1,0.506
49,-0.787,0.099,3.281,1.888,-1.200,-2.537,-0.834,0.699,0,1,0.888
106,-0.109,0.660,3.475,2.659,-0.764,-1.603,0.039,0.876,0,1,0.774
109,2.674,3.111,3.836,2.943,1.119,-0.311,1.458,0.013,0,1,0.806
122,-0.435,-0.895,2.209,1.282,0.431,-2.359,-0.575,-1.276,0,1,0.691
124,0.813,-1.690,1.692,1.094,-2.110,-3.447,-1.025,1.339,1,0,0.464
125,-1.859,-0.313,2.883,0.762,-1.096,-2.894,-1.427,-0.068,0,1,0.956
183,0.728,0.804,-0.860,-1.813,1.245,3.252,3.341,2.231,1,0,0.346
184,1.609,-2.039,2.058,1.993,0.461,-4.031,-1.032,-1.522,1,0,0.275



CLAIM REWRITE

TOO-STRONG CLAIM:

"The model accurately predicts the target and generalizes well."


EVIDENCE-SAFE CLAIM:

"Observed and measured: the model achieved 0.835 accuracy
and 0.924 ROC-AUC on the grouped stress-test split.

The result was directionally stable compared with the original
random split. These measurements support the model as
decision-support on this synthetic evaluation.

However, the results do not establish generalization to unseen
clients or future production data because the Week-5 dataset
does not contain real client IDs or timestamps."



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.